# 03 — Analyse exploratoire

À partir du fichier propre produit par `02_nettoyage.ipynb`. Les figures sont enregistrées
dans le dossier `figures/` pour être affichées dans le README.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

racine = Path.cwd()
while not (racine / "data" / "raw").exists() and racine != racine.parent:
    racine = racine.parent

chemin = racine / "data" / "processed" / "logements_paca.parquet"
if not chemin.exists():
    raise FileNotFoundError(f"Fichier introuvable : {chemin}\n→ Lance d'abord le notebook 02_nettoyage jusqu'au bout.")

logements = pd.read_parquet(chemin)

dossier_fig = racine / "figures"
dossier_fig.mkdir(exist_ok=True)

NOMS_DEP = {"04": "Alpes-de-Haute-Provence", "05": "Hautes-Alpes", "06": "Alpes-Maritimes",
            "13": "Bouches-du-Rhône", "83": "Var", "84": "Vaucluse"}
logements["code_departement"] = logements["code_departement"].astype(str).str.zfill(2)
logements["departement"] = logements["code_departement"].map(NOMS_DEP)

print(logements.shape)
logements["departement"].value_counts()

## 1. Distribution des prix au m²

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for type_local, couleur in [("Appartement", "tab:blue"), ("Maison", "tab:orange")]:
    s = logements.loc[logements["type_local"] == type_local, "prix_m2"]
    ax.hist(s, bins=100, alpha=0.5, color=couleur,
            label=f"{type_local} (médiane : {s.median():,.0f} €/m²)".replace(",", " "))
ax.set(title="Distribution des prix au m² en PACA (2021-2025)",
       xlabel="Prix au m² (€)", ylabel="Nombre de ventes")
ax.legend()
fig.savefig(dossier_fig / "distribution_prix_m2.png", dpi=150, bbox_inches="tight")
plt.show()

> ✍️ **À toi** — la forme de la distribution (symétrique ? longue traîne ?) et ce que ça implique pour le modèle.
>
> *(remplace ce bloc par ton commentaire)*

## 2. Carte des prix au m²

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
hb = ax.hexbin(
    logements["longitude"], logements["latitude"],
    C=logements["prix_m2"], reduce_C_function=np.median,  # prix médian dans chaque hexagone
    gridsize=150, mincnt=5,                                # au moins 5 ventes par hexagone
    cmap="magma", vmin=1500, vmax=8000,
)
fig.colorbar(hb, ax=ax, label="Prix médian au m² (€)")
ax.set_aspect(1 / np.cos(np.radians(43.8)))  # corrige la déformation due à la latitude
ax.set(title="Prix médian au m² en PACA (ventes 2021-2025)", xlabel="Longitude", ylabel="Latitude")
fig.savefig(dossier_fig / "carte_prix_m2.png", dpi=150, bbox_inches="tight")
plt.show()

> ✍️ **À toi** — où sont les zones les plus chères et les moins chères ? (littoral, arrière-pays, stations de ski...)
>
> *(remplace ce bloc par ton commentaire)*

## 3. Évolution dans le temps

In [ ]:
trimestre = logements["date_mutation"].dt.to_period("Q")
evolution = logements.groupby([trimestre, "type_local"])["prix_m2"].median().unstack()
volume = logements.groupby(trimestre).size()
evolution.index = evolution.index.to_timestamp()
volume.index = volume.index.to_timestamp()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for colonne in evolution.columns:
    ax1.plot(evolution.index, evolution[colonne], marker="o", label=colonne)
ax1.set(title="Prix médian au m² par trimestre", ylabel="€/m²")
ax1.legend()
ax2.bar(volume.index, volume.values, width=60)
ax2.set(title="Nombre de ventes par trimestre", ylabel="Ventes")
fig.savefig(dossier_fig / "evolution_temporelle.png", dpi=150, bbox_inches="tight")
plt.show()

> ✍️ **À toi** — décris les phases (hausse 2021-2023, baisse fin 2023-2024, reprise 2025), le lien possible avec les taux d'intérêt, et la conséquence pour découper les données entre entraînement et test.
>
> *(remplace ce bloc par ton commentaire)*

## 4. Prix par département

In [ ]:
ordre = logements.groupby("departement")["prix_m2"].median().sort_values().index

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, type_local in zip(axes, ["Appartement", "Maison"]):
    sous = logements[logements["type_local"] == type_local]
    ax.boxplot([sous.loc[sous["departement"] == d, "prix_m2"] for d in ordre], showfliers=False)
    ax.set_xticks(range(1, len(ordre) + 1))
    ax.set_xticklabels(ordre, rotation=30, ha="right")
    ax.set_title(type_local)
axes[0].set_ylabel("Prix au m² (€)")
fig.suptitle("Prix au m² par département")
fig.savefig(dossier_fig / "prix_par_departement.png", dpi=150, bbox_inches="tight")
plt.show()

> ✍️ **À toi** — le classement des départements, le cas des Hautes-Alpes, et l'écart de prix à l'intérieur d'un même département.
>
> *(remplace ce bloc par ton commentaire)*

## 5. Communes les plus et les moins chères

In [ ]:
par_commune = logements.groupby("nom_commune").agg(
    nb_ventes=("prix_m2", "size"),
    prix_m2_median=("prix_m2", "median"),
)
# On ne garde que les communes avec assez de ventes pour que la médiane soit fiable
fiables = par_commune[par_commune["nb_ventes"] >= 100].sort_values("prix_m2_median")
print(f"{len(fiables)} communes avec au moins 100 ventes")

In [ ]:
print("🔻 Les 10 moins chères")
fiables.head(10)

In [ ]:
print("🔺 Les 10 plus chères")
fiables.tail(10).iloc[::-1]

> ✍️ **À toi** — qu'ont en commun les communes les plus chères ? et les moins chères ?
>
> *(remplace ce bloc par ton commentaire)*

## 6. Surface et prix

In [ ]:
echantillon = logements.sample(min(20_000, len(logements)), random_state=42)  # 20 000 points suffisent pour voir la tendance

fig, ax = plt.subplots(figsize=(9, 6))
for type_local, couleur in [("Appartement", "tab:blue"), ("Maison", "tab:orange")]:
    s = echantillon[echantillon["type_local"] == type_local]
    ax.scatter(s["surface_reelle_bati"], s["valeur_fonciere"], s=3, alpha=0.3, color=couleur, label=type_local)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(title="Prix de vente en fonction de la surface", xlabel="Surface (m², échelle log)",
       ylabel="Prix (€, échelle log)")
ax.legend(markerscale=5)
fig.savefig(dossier_fig / "surface_vs_prix.png", dpi=150, bbox_inches="tight")
plt.show()

> ✍️ **À toi** — la relation surface-prix en échelle log, et ce qui explique qu'à surface égale le prix varie autant.
>
> *(remplace ce bloc par ton commentaire)*

## Conclusions pour la modélisation

> ✍️ **À toi** — résume en 3 ou 4 points ce que cette analyse t'apprend pour construire le modèle (localisation fine, date, log du prix, découpage temporel).
>
> *(remplace ce bloc par ton commentaire)*